In [ ]:
# FraudGuard — Model Explainability & Error Analysis

## Day 5

### Objective

Understand why the XGBoost model performs better than the Logistic Regression baseline and identify the types of transactions the model gets wrong.

### Day 4 Benchmark

- Model: XGBoost
- ROC-AUC: 0.9049
- PR-AUC: 0.5391
- Best tested threshold: 0.30
- Precision: 62.82%
- Recall: 44.36%
- F1: 0.5200
- Fraud caught: 1,602
- False positives: 948
- False negatives: 2,009

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

from scipy.sparse import load_npz
import joblib

PROJECT_ROOT = Path.cwd().parent
processed_dir = PROJECT_ROOT / "data" / "processed"

X_validation_final = load_npz(
    processed_dir / "X_validation.npz"
)

y_validation = pd.read_csv(
    processed_dir / "y_validation.csv"
).squeeze()

xgb_model = joblib.load(
    processed_dir / "xgboost_fraudguard.pkl"
)

feature_names = pd.read_csv(
    processed_dir / "feature_names.csv"
)["feature"].values

print("Validation matrix:", X_validation_final.shape)
print("Validation target:", y_validation.shape)
print("Features:", len(feature_names))
print("Model:", type(xgb_model))

Validation matrix: (105693, 891)
Validation target: (105693,)
Features: 891
Model: <class 'xgboost.sklearn.XGBClassifier'>


In [2]:
y_validation_proba = xgb_model.predict_proba(
    X_validation_final
)[:, 1]

threshold = 0.30

y_validation_pred = (
    y_validation_proba >= threshold
).astype(int)

print("Predictions generated.")
print("Fraud predictions:", y_validation_pred.sum())

Predictions generated.
Fraud predictions: 2550


In [4]:
validation_results = pd.DataFrame({
    "actual": y_validation.values,
    "probability": y_validation_proba,
    "prediction": y_validation_pred
})

validation_results["error_type"] = np.select(
    [
        (validation_results["actual"] == 0) &
        (validation_results["prediction"] == 0),

        (validation_results["actual"] == 0) &
        (validation_results["prediction"] == 1),

        (validation_results["actual"] == 1) &
        (validation_results["prediction"] == 0),

        (validation_results["actual"] == 1) &
        (validation_results["prediction"] == 1)
    ],
    [
        "True Negative",
        "False Positive",
        "False Negative",
        "True Positive"
    ],
    default="Unknown"
)

print(
    validation_results["error_type"].value_counts()
)

error_type
True Negative     101134
False Negative      2009
True Positive       1602
False Positive       948
Name: count, dtype: int64


In [5]:
print(
    validation_results["error_type"].value_counts()
)

error_type
True Negative     101134
False Negative      2009
True Positive       1602
False Positive       948
Name: count, dtype: int64


In [6]:
error_summary = (
    validation_results
    .groupby("error_type")["probability"]
    .agg(
        count="count",
        mean_probability="mean",
        median_probability="median",
        max_probability="max"
    )
)

display(error_summary)

,count,mean_probability,median_probability,max_probability
error_type,,,,
False Negative,2009,0.087339,0.060974,0.299391
False Positive,948,0.506101,0.444361,0.978035
True Negative,101134,0.020434,0.009646,0.299786
True Positive,1602,0.750998,0.831175,0.998172


In [7]:
false_negatives = validation_results[
    validation_results["error_type"] == "False Negative"
].copy()

print("False negatives:", len(false_negatives))

print("\nProbability statistics:")
print(
    false_negatives["probability"].describe()
)

False negatives: 2009

Probability statistics:
count    2009.000000
mean        0.087339
std         0.078355
min         0.001261
25%         0.022674
50%         0.060974
75%         0.135836
max         0.299391
Name: probability, dtype: float64


In [8]:
false_negatives["probability_bin"] = pd.cut(
    false_negatives["probability"],
    bins=[0, 0.05, 0.10, 0.15, 0.20, 0.25, 0.30],
    include_lowest=True
)

fn_probability_analysis = (
    false_negatives
    .groupby("probability_bin", observed=False)
    .size()
    .reset_index(name="fraud_count")
)

display(fn_probability_analysis)

,probability_bin,fraud_count
0,"(-0.001, 0.05]",899
1,"(0.05, 0.1]",421
2,"(0.1, 0.15]",256
3,"(0.15, 0.2]",189
4,"(0.2, 0.25]",136
5,"(0.25, 0.3]",108


In [9]:
from sklearn.metrics import precision_score, recall_score, f1_score

threshold_sensitivity = []

for threshold in [0.05, 0.10, 0.15, 0.20, 0.25, 0.30]:

    predictions = (
        y_validation_proba >= threshold
    ).astype(int)

    threshold_sensitivity.append({
        "threshold": threshold,
        "precision": precision_score(
            y_validation,
            predictions,
            zero_division=0
        ),
        "recall": recall_score(
            y_validation,
            predictions,
            zero_division=0
        ),
        "f1": f1_score(
            y_validation,
            predictions,
            zero_division=0
        ),
        "flagged": predictions.sum()
    })

threshold_sensitivity = pd.DataFrame(
    threshold_sensitivity
)

display(threshold_sensitivity)

,threshold,precision,recall,f1,flagged
0,0.05,0.213981,0.751038,0.333067,12674
1,0.10,0.346491,0.634450,0.448205,6612
2,0.15,0.441240,0.563556,0.494953,4612
3,0.20,0.516364,0.511216,0.513777,3575
4,0.25,0.572481,0.473553,0.518339,2987
5,0.30,0.628235,0.443644,0.520045,2550


In [10]:
hardest_fraud = validation_results[
    validation_results["error_type"] == "False Negative"
].sort_values(
    "probability"
)

display(
    hardest_fraud.head(20)
)

,actual,probability,prediction,error_type
75084,1,0.001261,0,False Negative
101025,1,0.001417,0,False Negative
35050,1,0.001745,0,False Negative
11202,1,0.001845,0,False Negative
20675,1,0.001929,0,False Negative
83821,1,0.002090,0,False Negative
11415,1,0.002138,0,False Negative
32270,1,0.002159,0,False Negative
18721,1,0.002164,0,False Negative
11952,1,0.002234,0,False Negative


In [11]:
easiest_missed_fraud = validation_results[
    validation_results["error_type"] == "False Negative"
].sort_values(
    "probability",
    ascending=False
)

display(
    easiest_missed_fraud.head(20)
)

,actual,probability,prediction,error_type
84054,1,0.299391,0,False Negative
78575,1,0.298810,0,False Negative
2423,1,0.298144,0,False Negative
737,1,0.297880,0,False Negative
103421,1,0.297803,0,False Negative
1720,1,0.297396,0,False Negative
94291,1,0.297282,0,False Negative
48371,1,0.296905,0,False Negative
53924,1,0.295133,0,False Negative
13906,1,0.294519,0,False Negative


In [12]:
borderline_fraud = validation_results[
    (validation_results["error_type"] == "False Negative") &
    (validation_results["probability"] >= 0.25)
].copy()

true_positive = validation_results[
    validation_results["error_type"] == "True Positive"
].copy()

print("Borderline missed fraud:", len(borderline_fraud))
print("True positives:", len(true_positive))

Borderline missed fraud: 108
True positives: 1602


In [13]:
print("Borderline FN probability:")
print(borderline_fraud["probability"].describe())

print("\nTrue Positive probability:")
print(true_positive["probability"].describe())

Borderline FN probability:
count    108.000000
mean       0.274354
std        0.013126
min        0.250597
25%        0.264909
50%        0.272858
75%        0.283572
max        0.299391
Name: probability, dtype: float64

True Positive probability:
count    1602.000000
mean        0.750998
std         0.220217
min         0.300287
25%         0.562886
50%         0.831175
75%         0.950359
max         0.998172
Name: probability, dtype: float64


In [17]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
DATA_RAW = PROJECT_ROOT / "data" / "raw"

print("Project root:", PROJECT_ROOT)
print("Raw data:", DATA_RAW)

Project root: /Users/ayushkumar/Desktop/Fraudguard
Raw data: /Users/ayushkumar/Desktop/Fraudguard/data/raw


In [18]:
train_transaction = pd.read_csv(
    DATA_RAW / "train_transaction.csv"
)

train_identity = pd.read_csv(
    DATA_RAW / "train_identity.csv"
)

full_train = train_transaction.merge(
    train_identity,
    on="TransactionID",
    how="left"
)

print("Full training data:", full_train.shape)

Full training data: (590540, 434)


In [19]:
SECONDS_PER_DAY = 24 * 60 * 60

full_train["transaction_day"] = (
    full_train["TransactionDT"] // SECONDS_PER_DAY
)

In [20]:
validation_original = full_train[
    full_train["transaction_day"] > 145
].copy()

print("Validation rows:", len(validation_original))

Validation rows: 105693


In [21]:
validation_original["predicted_probability"] = (
    y_validation_proba
)

validation_original["prediction"] = (
    y_validation_pred
)

In [22]:
borderline_mask = (
    (validation_original["isFraud"] == 1) &
    (validation_original["predicted_probability"] >= 0.25) &
    (validation_original["predicted_probability"] < 0.30)
)

borderline_original = validation_original[
    borderline_mask
].copy()

print(
    "Borderline fraud transactions:",
    len(borderline_original)
)

Borderline fraud transactions: 108


In [23]:
true_positive_mask = (
    (validation_original["isFraud"] == 1) &
    (validation_original["predicted_probability"] >= 0.30)
)

true_positive_original = validation_original[
    true_positive_mask
].copy()

print("True positives:", len(true_positive_original))

True positives: 1602


In [24]:
comparison_features = [
    "V258",
    "V294",
    "V70",
    "C7",
    "C4",
    "C14",
    "card6",
    "id_17",
    "id_30",
    "DeviceInfo"
]

comparison = pd.DataFrame({
    "feature": comparison_features,
    "borderline_FN_mean": [
        borderline_original[f].mean()
        if pd.api.types.is_numeric_dtype(borderline_original[f])
        else borderline_original[f].nunique()
        for f in comparison_features
    ],
    "TP_mean_or_unique": [
        true_positive_original[f].mean()
        if pd.api.types.is_numeric_dtype(true_positive_original[f])
        else true_positive_original[f].nunique()
        for f in comparison_features
    ]
})

display(comparison)

,feature,borderline_FN_mean,TP_mean_or_unique
0,V258,1.529412,4.151294
1,V294,7.768519,1.637328
2,V70,0.012500,0.021441
3,C7,0.527778,6.213483
4,C4,39.083333,18.330836
5,C14,28.824074,10.398252
6,card6,2.000000,2.000000
7,id_17,199.207547,209.408776
8,id_30,9.000000,28.000000
9,DeviceInfo,12.000000,103.000000


In [25]:
for feature in [
    "card6",
    "id_30",
    "DeviceInfo"
]:
    
    print("\n" + "=" * 60)
    print(feature)
    print("=" * 60)
    
    print("\nBorderline FN:")
    print(
        borderline_original[feature]
        .fillna("__MISSING__")
        .value_counts()
        .head(10)
    )
    
    print("\nTrue Positive:")
    print(
        true_positive_original[feature]
        .fillna("__MISSING__")
        .value_counts()
        .head(10)
    )


card6

Borderline FN:
card6
debit     61
credit    47
Name: count, dtype: int64

True Positive:
card6
credit         859
debit          733
__MISSING__     10
Name: count, dtype: int64

id_30

Borderline FN:
id_30
__MISSING__         88
Windows 10           5
Mac OS X 10_13_4     4
Windows Vista        2
iOS 11.3.0           2
Windows 7            2
Mac OS X 10_10_5     2
Mac OS X 10_11_6     1
Mac OS X 10_9_5      1
iOS 10.0.2           1
Name: count, dtype: int64

True Positive:
id_30
__MISSING__         1309
iOS 11.3.0            80
Windows 10            61
Android 7.0           17
Android 7.1.1         14
Windows 7             13
Mac OS X 10_9_5       10
Mac OS X 10_13_4       9
Mac OS X 10_10_5       9
iOS 11.0.3             7
Name: count, dtype: int64

DeviceInfo

Borderline FN:
DeviceInfo
__MISSING__                      67
Windows                          19
MacOS                             8
iOS Device                        3
CRO-L03 Build/HUAWEICRO-L03       2
Blade A510 B

In [26]:
v258_comparison = pd.DataFrame({
    "group": [
        "Borderline FN",
        "True Positive"
    ],
    "mean": [
        borderline_original["V258"].mean(),
        true_positive_original["V258"].mean()
    ],
    "median": [
        borderline_original["V258"].median(),
        true_positive_original["V258"].median()
    ],
    "missing": [
        borderline_original["V258"].isna().sum(),
        true_positive_original["V258"].isna().sum()
    ]
})

display(v258_comparison)

,group,mean,median,missing
0,Borderline FN,1.529412,1.0,57
1,True Positive,4.151294,3.0,366


In [32]:
def v258_groups(df):
    return (
        pd.cut(
            df["V258"],
            bins=[-np.inf, 1, 2, np.inf],
            labels=["0-1", "1-2", ">2"]
        )
        .value_counts(normalize=True)
        .sort_index()
        * 100
    )


v258_group_comparison = pd.DataFrame({
    "Borderline FN %": v258_groups(borderline_original),
    "True Positive %": v258_groups(true_positive_original)
})

display(v258_group_comparison)

,Borderline FN %,True Positive %
V258,,
0-1,62.745098,24.271845
1-2,25.490196,20.226537
>2,11.764706,55.501618


In [33]:
v258_summary = []

for group_name, df in [
    ("Borderline FN", borderline_original),
    ("True Positive", true_positive_original)
]:
    
    for group in ["0-1", "1-2", ">2"]:
        
        if group == "0-1":
            mask = df["V258"].between(0, 1, inclusive="right")
        elif group == "1-2":
            mask = (
                (df["V258"] > 1) &
                (df["V258"] <= 2)
            )
        else:
            mask = df["V258"] > 2
        
        subset = df[mask]
        
        v258_summary.append({
            "group": group_name,
            "V258_range": group,
            "transactions": len(subset),
            "mean_probability": subset[
                "predicted_probability"
            ].mean()
        })

v258_summary = pd.DataFrame(v258_summary)

display(v258_summary)

,group,V258_range,transactions,mean_probability
0,Borderline FN,0-1,32,0.274532
1,Borderline FN,1-2,13,0.280475
2,Borderline FN,>2,6,0.272415
3,True Positive,0-1,300,0.698592
4,True Positive,1-2,250,0.746464
5,True Positive,>2,686,0.862319
